# Web exploration and scraping

Inside the devcontainer, start `make demo`, choose the project `.venv` kernel, and run cells in order. The session uses the remote Playwright browser automatically. Notebooks are not included in the standalone test image. The demo needs no external website.


In [ ]:
import sys
from pathlib import Path

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from notebooks import NotebookSession

import os
from environment import load_environment

load_environment()
BASE_URL = os.environ.get("BASE_URL", "http://127.0.0.1:8765")  # Start `make demo` in another terminal.
from components.scraping import extract_links
from playwright.sync_api import expect


In [ ]:
session = await NotebookSession.web(BASE_URL)


## Reuse application components

Synchronous snippets from the Playwright CLI can run inside a normal callback. All Playwright operations stay on the worker thread; return plain values to notebook cells.


In [ ]:
def inspect_home(app):
    app.open()
    expect(app.heading).to_have_text("Example application")
    return {"title": app.page.title(), "url": app.page.url, "links": extract_links(app.page)}

data = await session.run(inspect_home)
assert data["links"]
display(data)


In [ ]:
def submit_contact(app):
    app.open("/contact")
    app.contact.fill(name="Alex", email="alex@example.com", subject="Sales")
    app.contact.submit()
    expect(app.contact.status).to_have_text("Message sent by Alex")
    return app.contact.status.inner_text()

message = await session.run(submit_contact)
display(message)


## Reuse scraping scripts and validate results

`extract_links(page)` in `src/components/scraping.py` is reusable by notebooks and tests. For another site, change the URL inside the callback and adapt extraction to its page structure.


In [ ]:
def scrape_links(app):
    app.open()
    return extract_links(app.page)

links = await session.run(scrape_links)
assert {row["text"] for row in links} == {"Home", "Contact", "Tasks"}
assert all(row["url"].startswith(BASE_URL) for row in links)
display(links)


In [ ]:
import csv
output = ROOT / "notes" / "output"
output.mkdir(exist_ok=True)
with (output / "links.csv").open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["text", "url"])
    writer.writeheader()
    writer.writerows(links)


In [ ]:
from IPython.display import Image
image = await session.run(lambda app: app.page.screenshot(full_page=True))
display(Image(data=image))


## Release browser resources

Run this even after a failed exploratory cell. Restarting setup requires a new session.


In [ ]:
await session.close()
